Prototype

In [7]:
!pip install spacy bs4 openai

In [16]:
! python -m spacy download en_core_web_sm python_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 18.2 MB/s eta 0:00:00a 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [26]:
import requests
from bs4 import BeautifulSoup
import spacy
import openai
import time

from openai import OpenAI
from dotenv import load_dotenv
import os
import os
from dotenv import load_dotenv
from openai import OpenAI

# ✅ Step 1: Load your .env file
load_dotenv()

# ✅ Step 2: Access the API key
api_key = os.getenv("OPENAI_API_KEY")

# ✅ Step 3: Initialize the OpenAI client with the key
client = OpenAI(api_key=api_key)

In [28]:


# Step 1: Download the article content
def fetch_webpage(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/115.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text

# NPR article URL
url = "https://www.npr.org/2020/01/27/799879297/billie-eilish-sweeps-grammys-in-ceremony-clouded-by-controversy-and-mourning?utm_source=chatgpt.com"
html_content = fetch_webpage(url)

In [34]:
models = client.models.list()
for model in models:
    print(model.id)


dall-e-3
dall-e-2
gpt-4o-audio-preview-2024-10-01
gpt-4o-transcribe
gpt-4o-mini-transcribe
babbage-002
gpt-4o-mini-tts
tts-1-hd-1106
text-embedding-3-large
text-embedding-ada-002
tts-1-hd
gpt-4o-mini-audio-preview
gpt-4o-audio-preview
o1-preview-2024-09-12
gpt-3.5-turbo-instruct-0914
gpt-4o-mini-search-preview
tts-1-1106
davinci-002
gpt-3.5-turbo-1106
gpt-3.5-turbo-instruct
gpt-3.5-turbo
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-2024-11-20
whisper-1
gpt-3.5-turbo-0125
gpt-4o-2024-05-13
gpt-3.5-turbo-16k
o1-preview
gpt-4o-search-preview
gpt-4.5-preview
gpt-4.5-preview-2025-02-27
gpt-4o-search-preview-2025-03-11
omni-moderation-latest
tts-1
omni-moderation-2024-09-26
text-embedding-3-small
gpt-4o
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-mini-2024-07-18
o1-mini
gpt-4o-mini-audio-preview-2024-12-17
o1-mini-2024-09-12


In [29]:
# Step 2: Extract the main article text using BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# NPR articles often use a div with itemprop="articleBody"
article_body = soup.find("div", {"itemprop": "articleBody"})
if article_body is None:
    # Fallback: try another likely container (you may need to adjust this)
    article_body = soup.find("div", {"class": "storytext"})

if article_body:
    paragraphs = article_body.find_all("p")
    article_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
else:
    article_text = soup.get_text()

print("Article text extracted (first 500 characters):")
print(article_text[:500])

Article text extracted (first 500 characters):
Billie Eilish, who swept the four major categories at the 2020 Grammy Awards, accepts her trophy for Song of the Year, for "Bad Guy."Robyn Beck/AFP via Getty Imageshide caption
Billie Eilish, who swept the four major categories at the 2020 Grammy Awards, accepts her trophy for Song of the Year, for "Bad Guy."
By the end of the 62nd Grammy Awards on Sunday evening, a major star had been crowned: 18-year-old singer Billie Eilish, who swept all four of the night's biggest prizes — Best New Artist, 


In [30]:

# Step 3: Set up spaCy and perform Named Entity Recognition (NER)
# Make sure to install spaCy and download the model with:
# pip install spacy
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")
doc = nlp(article_text)

print("\n=== Named Entities Extracted by spaCy ===")
for ent in doc.ents:
    print(f"{ent.text} ({ent.label_})")



=== Named Entities Extracted by spaCy ===
Billie Eilish (PERSON)
four (CARDINAL)
the 2020 Grammy Awards (EVENT)
Beck/AFP (ORG)
Getty Imageshide (ORG)
four (CARDINAL)
the 2020 Grammy Awards (EVENT)
the end of the 62nd (DATE)
Sunday (DATE)
evening (TIME)
18-year-old (DATE)
Billie Eilish (PERSON)
four (CARDINAL)
night (TIME)
the Year and Album of the Year (DATE)
Sunday night (TIME)
the Staples Center (FAC)
Los Angeles (GPE)
evening (TIME)
the Recording Academy (ORG)
just a few weeks ago (DATE)
Los Angeles (GPE)
Bryantin (GPE)
just hours (TIME)
the Recording Academy (ORG)
year (DATE)
Academy (ORG)
Deborah Dugan (PERSON)
earlier this month (DATE)
recent years (DATE)
Beyoncé (ORG)
Kendrick Lamar (PERSON)
Grammys (GPE)
Academy (ORG)
2020 (DATE)
Lizzo (WORK_OF_ART)
Lil Nas X (PERSON)
Ariana Grande (ORG)
H.E.R. (ORG)
Lana Del Rey (PERSON)
Lizzo (PERSON)
the night (TIME)
five (CARDINAL)
Grammys (GPE)
the Album of the Year (WORK_OF_ART)
Year (DATE)
first (ORDINAL)
second (ORDINAL)
Christopher Cr

In [36]:

# Step 4: Use an LLM to perform relationship extraction based on the text.
# Define a function that uses OpenAI's API.
# Ensure your OpenAI API key is set (e.g., via the OPENAI_API_KEY environment variable).
def extract_relationships_via_llm(text):
    prompt = f"""
Extract RDF-style triples from the following text.
Use the format: Subject | Predicate | Object

Examples:
"Lady Gaga performed 'Shallow' at the Oscars in 2019."
Output:
Lady Gaga | performed | Shallow
Shallow | performedAt | Oscars_2019

"Kendrick Lamar released 'To Pimp a Butterfly' in 2015."
Output:
Kendrick Lamar | released | ToPimpAButterfly
ToPimpAButterfly | releaseYear | 2015

Now extract from:
"{text}"
    """
    # Call the OpenAI ChatCompletion API (adjust model and parameters as needed)
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo" if you're using that
        messages=[
            {"role": "system", "content": "You are an expert in extracting RDF triples from music-related articles."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return response.choices[0].message.content
# For demonstration, use the first 1000 characters of the article text
llm_input = article_text[:1000]
relationships = extract_relationships_via_llm(llm_input)

print("\n=== Extracted Relationships (via LLM) ===")
print(relationships)



=== Extracted Relationships (via LLM) ===
Billie Eilish | swept | four major categories
Billie Eilish | accepts | trophy for Song of the Year
Song of the Year | awardedFor | Bad Guy
Billie Eilish | won | Best New Artist
Billie Eilish | won | Song of the Year
Billie Eilish | won | Record of the Year
Billie Eilish | won | Album of the Year
Billie Eilish | won | Best Pop Vocal Album
62nd Grammy Awards | heldAt | Staples Center
62nd Grammy Awards | tookPlaceOn | Sunday evening
62nd Grammy Awards | heldIn | Los Angeles
